# Navier–Stokes pipe flow

*fluids · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/navier_stokes_pipe_flow)

The full incompressible Navier–Stokes equations, applied to water in a round pipe that is suddenly pushed by a pressure gradient. For axial flow the nonlinear term drops out exactly, leaving a radial diffusion equation: momentum spreads in from the no-slip wall and the profile grows into the Hagen–Poiseuille parabola within a few times $R^2/\nu$.

**Vector form**

$$\rho\big(\partial_t\mathbf u + (\mathbf u\cdot\nabla)\mathbf u\big) = -\nabla p + \mu\,\nabla^2\mathbf u,\qquad \nabla\cdot\mathbf u = 0$$

**Index (tensor) form**

$$\rho\,(\partial_t u_i + u_j\,\partial_j u_i) = -\partial_i p + \mu\,\partial_j\partial_j u_i,\quad \partial_i u_i = 0 \qquad \xrightarrow{\ \mathbf u = u_z(r,t)\,\hat{\mathbf z}\ }\qquad \partial_t u_z = \frac{G}{\rho} + \frac{\nu}{r}\,\partial_r(r\,\partial_r u_z),\quad \dot u_i = \frac G\rho + \nu\,L_{ij}u_j$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_j\,\partial_j u_i$ | `(vanishes)` | — | only $u_z \ne 0$ and $\partial_z u_z = 0$, so the advection term is exactly zero |
| $\partial_i u_i$ | `(vanishes)` | — | continuity holds automatically: $u_z$ does not change along the pipe |
| $r_i$ | `grid(N, R, dev)` | (N+1,) | radial points from the axis ($i = 0$) to the wall ($i = N$) |
| $u_i$ | `u` | (N,) | axial velocity $u_z(r_i)$; the wall value $u_N = 0$ (no slip) |
| $L_{ij}$ | `radial_laplacian(r, dr)` | (N, N) | $\tfrac1r\partial_r(r\partial_r\,\cdot)$ as a matrix, with the $r \to 0$ limit $2\partial_r^2$ on the axis |
| $L_{ij}\,u_j$ | `einsum('ij,j->i', L, u)` | (N,) | repeated $j$ ⇒ summed: viscous term $\nu\nabla^2 u_z$ |
| $G = -\partial_z p$ | `params['G']` | scalar | driving pressure gradient in Pa/m |
| $Q = \int 2\pi r\,u_z\,dr$ | `flow_rate(u, r)` | (T,) | volume flow rate; steady value $\pi G R^4/8\mu$ |
| $\tau_w = -\mu\,\partial_r u_z\vert _R$ | `wall_shear_stress(u, dr, mu)` | (T,) | drag per area on the wall; steady value $GR/2$ |

This is one of the rare exact solutions of Navier–Stokes: because the flow is parallel, $(\mathbf u\cdot\nabla)\mathbf u = 0$ and only the pressure gradient and viscosity remain. The steady Hagen–Poiseuille flow has $u_z = G(R^2 - r^2)/4\mu$, flow rate $Q = \pi G R^4/8\mu$ (the famous $R^4$ law) and wall shear stress $\tau_w = GR/2$ — the force balance $G\,\pi R^2 = \tau_w\,2\pi R$ per unit length. The start-up approaches it like $e^{-\lambda_1^2\nu t/R^2}$ with $\lambda_1 = 2.405$ (first zero of $J_0$), so by $t \approx 0.6\,R^2/\nu$ the profile is within a few percent of the parabola. Early on the core moves as a plug and only the fluid near the wall feels the drag — watch the dye lines. The solution is laminar; above a Reynolds number of about 2000 real pipe flow becomes turbulent.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/navier_stokes_pipe_flow/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import flow_rate, hagen_poiseuille, wall_shear_stress

N, RHO = 40, 1000.0  # radial grid intervals, density of water (kg/m³)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    R = params["R"] / 1000  # pipe radius in m
    nu = params["nu"] * 1e-6  # kinematic viscosity in m²/s
    G, mu = params["G"], RHO * nu
    tau = R**2 / nu  # viscous diffusion time across the pipe (s)

    r, dr = tensors.grid(N, R, dev)  # (N+1,) from axis to wall
    L = tensors.radial_laplacian(r, dr)  # (N, N)
    times, u = solve.simulate(tensors.at_rest(N, dev), L, G, RHO, nu, dr, params["t_max"] * tau)  # (T,), (T, N+1)
    u_s = hagen_poiseuille(r, G, mu, R)  # (N+1,)
    releases, C = solve.dye_lines(times, u, release_every=0.02 * tau)

    Q, Q_s = flow_rate(u[-1], r), math.pi * G * R**4 / (8 * mu)
    print(
        f"R²/ν = {tau:.1f} s,  {len(times) - 1} RK4 steps,  "
        f"Reynolds number 2R·u_mean/ν = {2 * Q_s / (math.pi * R * nu):.1f}"
    )
    print(f"centreline speed: {u[-1, 0] * 1000:.3f} mm/s  (Hagen–Poiseuille G R²/4μ = {u_s[0] * 1000:.3f} mm/s)")
    print(f"flow rate: {Q * 1e9:.3f} mm³/s  (Hagen–Poiseuille π G R⁴/8μ = {Q_s * 1e9:.3f} mm³/s)")
    print(
        f"wall shear stress: {wall_shear_stress(u[-1], dr, mu) * 1000:.4f} mPa  "
        f"(steady G R/2 = {G * R / 2 * 1000:.4f} mPa)"
    )
    k63 = int((u[:, 0] > 0.63 * u_s[0]).int().argmax())
    print(f"centreline reaches 63 % of its final speed at t = {times[k63] / tau:.3f} R²/ν")
    return plot.animate(times, r, u, C, releases, u_s, R, tau, out_dir / "navier_stokes_pipe_flow.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/navier_stokes_pipe_flow/equation.py
r"""Physics → tensor translation.

The incompressible Navier–Stokes equations, in index form (repeated j summed):

    ρ (∂_t u_i + u_j ∂_j u_i) = -∂_i p + μ ∂_j ∂_j u_i,        ∂_i u_i = 0

Pipe flow: fluid at rest in a round pipe of radius R is suddenly pushed by a pressure gradient
G = -∂p/∂z. The flow stays axial, u = u_z(r, t) \hat z, so

    ∂_i u_i = ∂_z u_z = 0           (u_z does not depend on z: continuity holds automatically)
    u_j ∂_j u_z = u_z ∂_z u_z = 0   (the nonlinear advection term vanishes!)

and the z-component of Navier–Stokes, with the Laplacian in cylindrical coordinates, becomes linear:

    ∂_t u_z = G/ρ + ν (1/r) ∂_r (r ∂_r u_z),       u_z(R) = 0 (no slip),  ν = μ/ρ

On grid points r_i = i Δr the radial Laplacian is a matrix L_{ij}, so the PDE is

    \dot u_i = G/ρ + ν L_{ij} u_j        (j summed → einsum('ij,j->i'))

Its steady state is the Hagen–Poiseuille profile u_s = G (R^2 - r^2) / (4μ).
"""

import math

import torch


def rhs(u: torch.Tensor, L: torch.Tensor, G: float, rho: float, nu: float) -> torch.Tensor:
    r"""\dot u_i = G/ρ + ν L_{ij} u_j.   u: (N,), L: (N, N) → (N,)"""
    return G / rho + nu * torch.einsum("ij,j->i", L, u)


def hagen_poiseuille(r: torch.Tensor, G: float, mu: float, R: float) -> torch.Tensor:
    r"""Steady profile u_s = G (R^2 - r^2) / (4μ).   (N,) → (N,)"""
    return G * (R**2 - r**2) / (4 * mu)


def flow_rate(u: torch.Tensor, r: torch.Tensor) -> torch.Tensor:
    r"""Q = ∫_0^R 2π r u dr (trapezoid rule on the grid).   u: (..., N), r: (N,) → (...)"""
    return torch.trapezoid(2 * math.pi * r * u, r, dim=-1)


def wall_shear_stress(u: torch.Tensor, dr: float, mu: float) -> torch.Tensor:
    r"""τ_w = -μ ∂_r u_z at r = R, from a one-sided 2nd-order difference (u at the wall is 0).   (..., N) → (...)"""
    wall, u1, u2 = u[..., -1], u[..., -2], u[..., -3]
    return -mu * (3 * wall - 4 * u1 + u2) / (2 * dr)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/navier_stokes_pipe_flow/tensors.py
"""Build the concrete tensors: radial grid points and the cylindrical Laplacian matrix."""

import torch


def grid(n: int, R: float, device: torch.device) -> tuple[torch.Tensor, float]:
    """N + 1 radial points r_i = i Δr from the axis (i = 0) to the wall (i = N), and Δr = R/N."""
    dr = R / n
    return dr * torch.arange(n + 1, device=device, dtype=torch.float64), dr


def radial_laplacian(r: torch.Tensor, dr: float) -> torch.Tensor:
    """(N, N) matrix for (1/r) ∂_r(r ∂_r u) acting on the unknowns u_0 … u_{N-1}; the wall value u_N = 0.

    Interior i:  [r_{i+½}(u_{i+1} - u_i) - r_{i-½}(u_i - u_{i-1})] / (r_i Δr²)
    Axis i = 0:  the symmetric limit 2 ∂_r² u = 4 (u_1 - u_0) / Δr²
    Row N-1 simply drops its u_N term, because u_N = 0 (no slip).
    """
    n = len(r) - 1
    L = torch.zeros(n, n + 1, dtype=r.dtype, device=r.device)  # one extra column for the wall, removed below
    L[0, 0], L[0, 1] = -4 / dr**2, 4 / dr**2
    for i in range(1, n):
        outer, inner = r[i] + dr / 2, r[i] - dr / 2  # r_{i+½}, r_{i-½}
        L[i, i + 1] = outer / (r[i] * dr**2)
        L[i, i - 1] = inner / (r[i] * dr**2)
        L[i, i] = -(outer + inner) / (r[i] * dr**2)
    return L[:, :n]


def at_rest(n: int, device: torch.device) -> torch.Tensor:
    """The unknowns u_0 … u_{N-1} are all 0 at t = 0: shape (N,)."""
    return torch.zeros(n, device=device, dtype=torch.float64)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/navier_stokes_pipe_flow/solve.py
"""March the start-up flow with RK4; also the exact Bessel-series solution and the dye-line positions."""

import math

import torch

from problems._common.integrators import integrate

from .equation import rhs


def simulate(
    u0: torch.Tensor, L: torch.Tensor, G: float, rho: float, nu: float, dr: float, t_max: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), u (T, N+1)) with the wall value u_N = 0 appended.

    Explicit RK4 is stable for diffusion while λ_max Δt < 2.79; here λ_max ≲ 8ν/Δr², so Δt = Δr²/(4ν) is safe.
    """
    dt = dr**2 / (4 * nu)
    n_steps = int(math.ceil(t_max / dt))
    u = integrate(lambda u, t: rhs(u, L, G, rho, nu), u0, dt, n_steps)  # (T, N)
    u = torch.nn.functional.pad(u, (0, 1))  # (T, N+1): add u = 0 at the wall
    return torch.arange(n_steps + 1, device=u0.device, dtype=u0.dtype) * dt, u


def bessel_zeros(n: int) -> torch.Tensor:
    """First n positive zeros λ_k of J_0, by Newton's method from the estimate (k - ¼)π (J_0' = -J_1)."""
    lam = (torch.arange(1, n + 1, dtype=torch.float64) - 0.25) * math.pi
    for _ in range(20):
        lam = lam + torch.special.bessel_j0(lam) / torch.special.bessel_j1(lam)
    return lam


def series_solution(r: torch.Tensor, t: float, G: float, mu: float, rho: float, R: float, terms: int = 40):
    r"""Exact start-up flow in a pipe:

    u = G(R² - r²)/(4μ) - (2 G R²/μ) Σ_k J_0(λ_k r/R) / (λ_k³ J_1(λ_k)) exp(-λ_k² ν t / R²)
    """
    lam = bessel_zeros(terms).to(r.device)  # (K,)
    decay = torch.exp(-(lam**2) * (mu / rho) * t / R**2) / (lam**3 * torch.special.bessel_j1(lam))  # (K,)
    modes = torch.special.bessel_j0(r[:, None] * lam / R)  # (N, K)
    return G * (R**2 - r**2) / (4 * mu) - 2 * G * R**2 / mu * torch.einsum("nk,k->n", modes, decay)


def dye_lines(times: torch.Tensor, u: torch.Tensor, release_every: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Dye lines released from a wire across the pipe every `release_every` seconds (the hydrogen-bubble trick).

    C(t, r) = ∫_0^t u dt' is how far fluid at radius r has moved. A line released at t_k sits at
    z_k(r, t) = C(t, r) - C(t_k, r). Returns release times (K,) and the cumulative distance C (T, N+1).
    """
    dt = (times[1] - times[0]).item()
    C = torch.cumsum(u, dim=0) * dt - u * dt  # (T, N+1) left-Riemann sum, C(0) = 0
    releases = torch.arange(0.0, times[-1].item(), release_every, dtype=times.dtype)
    return releases, C


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/navier_stokes_pipe_flow/plot.py
"""Left: dye lines across the pipe bend into the Hagen–Poiseuille parabola. Right: the velocity profile u_z(r, t)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def mirror(a: np.ndarray) -> np.ndarray:
    """Values on r = 0 … R → values across the whole diameter, r = -R … R (the flow is axisymmetric)."""
    return np.concatenate([a[..., :0:-1], a], axis=-1)


def animate(
    times: torch.Tensor, r: torch.Tensor, u: torch.Tensor, C: torch.Tensor, releases: torch.Tensor,
    u_steady: torch.Tensor, R: float, tau: float, path: Path,
) -> Path:  # fmt: skip
    t = times.cpu().numpy()
    mm = 1000  # plot in millimetres
    ys = mirror(r.cpu().numpy()) * mm
    ys[: len(ys) // 2] *= -1  # left half of the diameter is negative
    us, cs, us_s = (mirror(a.cpu().numpy()) * mm for a in (u, C, u_steady))
    k_release = np.searchsorted(t, releases.cpu().numpy())  # time step of each release

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(8, 3.6), gridspec_kw={"width_ratios": [1.6, 1]})
    window = us_s.max() * 0.12 * tau  # how far the steady centreline flow moves in 0.12 R²/ν
    ax.set_xlim(-0.03 * window, window)
    ax.set_ylim(-R * mm * 1.15, R * mm * 1.15)
    ax.fill_between([-window, 2 * window], -R * mm, R * mm, color="#eff6ff")
    for yw in (-R * mm, R * mm):
        ax.axhline(yw, color="#374151", lw=3)
    ax.axvline(0, color="#9ca3af", lw=1, ls="--")
    ax.set_xlabel("z (mm)   dashed: bubble wire")
    ax.set_ylabel("r (mm)")
    ax.grid(False)
    lines = [ax.plot([], [], color="#2563eb", lw=1.5)[0] for _ in k_release]
    title = ax.set_title(" ")

    ax2.plot(us_s, ys, color="#111827", ls="--", lw=1, label="Hagen–Poiseuille")
    profile = ax2.plot(us[0], ys, color="#dc2626", lw=2, label="u_z(r, t)")[0]
    ax2.set_xlim(0, us_s.max() * 1.1)
    ax2.set_ylim(-R * mm * 1.15, R * mm * 1.15)
    ax2.set_xlabel("u_z (mm/s)")
    ax2.legend(loc="lower right", fontsize=7)

    def update(step: int):
        for line, k in zip(lines, k_release):
            if k <= step:
                line.set_data(cs[step] - cs[k], ys)  # z_k(r, t) = C(t, r) - C(t_k, r)
            else:
                line.set_data([], [])
        profile.set_xdata(us[step])
        title.set_text(f"t = {t[step]:.1f} s = {t[step] / tau:.3f} R²/ν")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 75), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'G': 0.5,
    'R': 5.0,
    'nu': 1.0,
    't_max': 0.6
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.navier_stokes_pipe_flow")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.navier_stokes_pipe_flow.main")
problem_dir = Path("problems/fluids/navier_stokes_pipe_flow")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
